In [14]:
import os
import numpy as np
import faiss

from dotenv import load_dotenv
from pypdf import PdfReader
from google import genai
from google.genai import types

In [15]:
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found. "
        "Add it to your .env file."
    )

In [17]:
PDF_FILE = "document.pdf"


if not os.path.exists(PDF_FILE):
    raise FileNotFoundError(
        f"{PDF_FILE} not found. "
        "Put your PDF inside the Day11 folder."
    )

In [18]:
print("=" * 60)
print("📄 Reading PDF...")
print("=" * 60)


reader = PdfReader(PDF_FILE)

pdf_text = ""


for page_number, page in enumerate(
    reader.pages,
    start=1
):

    text = page.extract_text()

    if text:
        pdf_text += text + "\n\n"


print(f"📚 Pages: {len(reader.pages)}")
print(f"📝 Characters extracted: {len(pdf_text)}")

📄 Reading PDF...


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FLSPCD+BebasNeueBold', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(1202, 0, 2465942020800), '/LastChar': 88, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [155, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 396, 396, 396, 396, 396, 396, 396, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 407, 0, 386, 0, 368, 344, 390, 0, 190, 0, 0, 0, 0, 427, 396, 0, 0, 402, 374, 356, 398, 402, 0, 430]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FLSPCD+BebasNeueBold', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(1202, 0, 2465942020800), '/LastChar': 88, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [155, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 396, 396, 396, 396, 396, 396, 3

📚 Pages: 60
📝 Characters extracted: 79161


In [19]:
def create_chunks(text, chunk_size=1000):

    words = text.split()

    chunks = []

    current_chunk = []

    current_length = 0


    for word in words:

        current_chunk.append(word)

        current_length += len(word) + 1


        if current_length >= chunk_size:

            chunk = " ".join(current_chunk)

            chunks.append(chunk)

            current_chunk = []

            current_length = 0


    if current_chunk:

        chunks.append(
            " ".join(current_chunk)
        )


    return chunks


chunks = create_chunks(pdf_text)


print(f"✂️ Total chunks: {len(chunks)}")

✂️ Total chunks: 76


In [20]:
print("\n🔄 Creating embeddings...")


embedding_response = client.models.embed_content(

    model="gemini-embedding-001",

    contents=chunks,

    config=types.EmbedContentConfig(

        task_type="RETRIEVAL_DOCUMENT"

    )
)


embeddings = [

    embedding.values

    for embedding
    in embedding_response.embeddings

]


🔄 Creating embeddings...


In [21]:
vectors = np.array(
    embeddings,
    dtype="float32"
)


print(
    f"🔢 Embedding shape: "
    f"{vectors.shape}"
)

🔢 Embedding shape: (76, 3072)


In [22]:
dimension = vectors.shape[1]


index = faiss.IndexFlatL2(
    dimension
)


In [23]:
index.add(vectors)


print(
    f"🗂️ Vectors stored in FAISS: "
    f"{index.ntotal}"
)


🗂️ Vectors stored in FAISS: 76


In [24]:
def search_pdf(query, top_k=3):

    # --------------------------------------------------------
    # Create embedding for user's question
    # --------------------------------------------------------

    query_response = client.models.embed_content(

        model="gemini-embedding-001",

        contents=query,

        config=types.EmbedContentConfig(

            task_type="RETRIEVAL_QUERY"

        )
    )


    query_vector = np.array(

        [
            query_response.embeddings[0].values
        ],

        dtype="float32"

    )


    # --------------------------------------------------------
    # Search FAISS
    # --------------------------------------------------------

    distances, indices = index.search(

        query_vector,

        top_k

    )


    # --------------------------------------------------------
    # Collect retrieved chunks
    # --------------------------------------------------------

    results = []


    for distance, idx in zip(

        distances[0],

        indices[0]

    ):

        if idx == -1:
            continue


        results.append({

            "text": chunks[idx],

            "distance": float(distance),

            "chunk": int(idx)

        })


    return results

In [26]:
def generate_answer(query, results):

    context = "\n\n".join(

        result["text"]

        for result in results

    )


    prompt = f"""
You are a helpful PDF question-answering assistant.

Answer the user's question using ONLY the
information provided in the context.

If the answer cannot be found in the context,
say:

"I couldn't find this information in the PDF."

Do not invent information.

Keep the answer clear and concise.

---------------- CONTEXT ----------------

{context}

-------------- END CONTEXT --------------

User Question:

{query}
"""


    response = client.models.generate_content(

        model="gemini-3.5-flash-lite",

        contents=prompt

    )


    return response.text


In [27]:
print("\n" + "=" * 60)
print("🤖 Day 11 — PDF RAG with Embeddings + FAISS")
print("=" * 60)

print("\nAsk questions about your PDF.")
print("Type 'exit' to quit.")

print("-" * 60)


🤖 Day 11 — PDF RAG with Embeddings + FAISS

Ask questions about your PDF.
Type 'exit' to quit.
------------------------------------------------------------


In [36]:
query = input("\nYou: ").strip()

results = search_pdf(query, top_k=3 )

for i, result in enumerate(

            results,

            start=1

        ):

            print(
                f"\n[{i}] "
                f"Chunk {result['chunk']} "
                f"| Distance: "
                f"{result['distance']:.4f}"
            )

            print(
                result["text"][:500]
            )



[1] Chunk 11 | Distance: 0.3980
and excess inventory. 1. Retail � Predictive Maintenance: AI agents monitor equipment health, predict potential failures, and schedule maintenance, reducing downtime and repair costs. � Supply Chain Optimization: AI manages inventory, tracks shipments, and adjusts delivery routes in real-time, improving operational efficiency and reducing costs. 2. Manufacturing � Patient Monitoring: AI agents track patient vitals and alert healthcare providers about critical changes, enabling faster response ti

[2] Chunk 13 | Distance: 0.4169
of assignments and exams, saving educators time and ensuring consistency in evaluation. � Network Optimization: AI monitors network performance in real-time, detecting issues and automatically adjusting resources to maintain service quality. � Customer Support Automation: AI chatbots provide real-time support, resolving customer issues and reducing wait times for service requests. � Project Scheduling and Management: AI agents op

In [37]:
answer = generate_answer(query, results)
print("\n🤖 AI:")

print(answer)


🤖 AI:
Based on the provided context, the use cases of agentic AI across various industry verticals include:

* **Retail:** 
  * Predictive Maintenance (monitoring equipment health, predicting failures, scheduling maintenance)
  * Supply Chain Optimization (managing inventory, tracking shipments, adjusting delivery routes in real-time)
  * Personalized shopping assistants, chatbots for customer support, inventory management AI, visual search systems, and recommendation engines.
* **Manufacturing:** 
  * Patient Monitoring (tracking patient vitals and alerting healthcare providers) *(Note: categorized under Manufacturing in the text)*
  * Personalized Treatment Plans (analyzing patient data to suggest tailored treatment options) *(Note: categorized under Manufacturing in the text)*
* **Healthcare:** 
  * Drug Discovery (sifting through datasets to identify drug candidates) *(Note: categorized under Healthcare in the text)*
  * Gene Editing (simulating the effects of gene edits) *(Note: 